In [6]:
import pandas as pd
path = "../data/processed/cleaned.csv"
data1 = pd.read_csv(path)
print(data1.shape)


(590540, 108)


In [7]:
print("\nTarget distribution:")
print(data1["isFraud"].value_counts())

print("\nTarget percentage:")
print(data1["isFraud"].value_counts(normalize=True) * 100)

print("\nNumerical columns:", len(
    data1.select_dtypes(include=["int64", "float64"]).columns
))

print("\nCategorical columns:", len(
    data1.select_dtypes(include=["object"]).columns
))


Target distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Target percentage:
isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64

Numerical columns: 84

Categorical columns: 24


In [8]:
categorical_cols = data1.select_dtypes(include=["object"]).columns

cardinality = (
    data1[categorical_cols]
    .nunique()
    .sort_values(ascending=False)
)

print(cardinality)

id_31            131
R_emaildomain     61
P_emaildomain     60
ProductCD          5
card6              5
card4              5
id_15              4
M4                 4
id_37              3
id_36              3
id_35              3
id_29              3
id_28              3
id_16              3
M8                 3
id_12              3
M9                 3
M7                 3
M6                 3
M5                 3
M3                 3
M2                 3
M1                 3
id_38              3
dtype: int64


In [9]:
print("TransactionDT" in data1.columns)

True


In [10]:
data1["TransactionHour"] = (
    data1["TransactionDT"] // 3600
) % 24

data1["TransactionDay"] = (
    data1["TransactionDT"] // (24 * 3600)
)

data1["TransactionWeek"] = (
    data1["TransactionDay"] // 7
)

data1["TransactionWeekday"] = (
    data1["TransactionDay"] % 7
)

In [11]:
print(
    data1[
        [
            "TransactionDT",
            "TransactionHour",
            "TransactionDay",
            "TransactionWeek",
            "TransactionWeekday"
        ]
    ].head()
)

   TransactionDT  TransactionHour  TransactionDay  TransactionWeek  \
0          86400                0               1                0   
1          86401                0               1                0   
2          86469                0               1                0   
3          86499                0               1                0   
4          86506                0               1                0   

   TransactionWeekday  
0                   1  
1                   1  
2                   1  
3                   1  
4                   1  


In [12]:
import numpy as np

data1["TransactionAmt_Log"] = np.log1p(
    data1["TransactionAmt"]
)

In [13]:
print(
    data1[
        ["TransactionAmt", "TransactionAmt_Log"]
    ].describe()
)

       TransactionAmt  TransactionAmt_Log
count   590540.000000       590540.000000
mean       135.027176            4.382960
std        239.162522            0.937183
min          0.251000            0.223943
25%         43.321000            3.791459
50%         68.769000            4.245190
75%        125.000000            4.836282
max      31937.391000           10.371564


In [14]:
print(
    data1.groupby("TransactionHour")["isFraud"]
    .mean()
    .sort_values(ascending=False)
)

TransactionHour
7     0.106102
8     0.093014
9     0.089956
6     0.077743
5     0.070302
10    0.053212
4     0.051890
11    0.038816
3     0.038314
2     0.037483
23    0.036997
18    0.035231
19    0.034738
20    0.034273
21    0.034005
22    0.032694
17    0.031530
0     0.031380
1     0.031314
12    0.030439
16    0.029511
15    0.025399
14    0.024216
13    0.022889
Name: isFraud, dtype: float64


In [15]:
print(
    data1.groupby("TransactionWeekday")["isFraud"]
    .mean()
)

TransactionWeekday
0    0.037174
1    0.036040
2    0.037115
3    0.035644
4    0.031452
5    0.033048
6    0.034514
Name: isFraud, dtype: float64


In [16]:
print(
    data1.groupby("TransactionAmt_Log")["isFraud"]
    .mean()
    .head(20)
)

TransactionAmt_Log
0.223943    0.0
0.240590    0.0
0.256191    1.0
0.300105    1.0
0.310422    0.0
0.353470    1.0
0.383219    1.0
0.394741    1.0
0.404131    0.0
0.404798    0.0
0.433729    0.0
0.451076    0.0
0.459322    1.0
0.462475    0.0
0.479335    0.0
0.522952    0.0
0.562469    0.0
0.564177    0.0
0.570980    0.0
0.579978    0.0
Name: isFraud, dtype: float64


In [17]:
data1["TransactionAmt_Bin"] = pd.qcut(
    data1["TransactionAmt"],
    q=10,
    duplicates="drop"
)

amount_fraud = (
    data1.groupby(
        "TransactionAmt_Bin",
        observed=True
    )["isFraud"]
    .agg(["count", "mean"])
)

print(amount_fraud)

                      count      mean
TransactionAmt_Bin                   
(0.25, 25.95]         59511  0.055889
(25.95, 35.95]        61650  0.032052
(35.95, 49.0]         65116  0.032250
(49.0, 57.95]         59647  0.019431
(57.95, 68.769]       49346  0.028513
(68.769, 100.0]       73349  0.036170
(100.0, 117.0]        72079  0.019742
(117.0, 159.95]       32399  0.043026
(159.95, 275.293]     58390  0.038037
(275.293, 31937.391]  59053  0.050870


In [18]:
categorical_check = [
    "ProductCD",
    "card4",
    "card6",
    "P_emaildomain",
    "R_emaildomain",
    "id_31"
]

for col in categorical_check:
    print(f"\n========== {col} ==========")

    result = (
        data1.groupby(col)["isFraud"]
        .agg(["count", "mean"])
        .sort_values("mean", ascending=False)
    )

    print(result.head(20))


========== ProductCD ==========
            count      mean
ProductCD                  
C           68519  0.116873
S           11628  0.058996
H           33024  0.047662
R           37699  0.037826
W          439670  0.020399

========== card4 ==========
                   count      mean
card4                             
discover            6651  0.077282
visa              384767  0.034756
mastercard        189217  0.034331
american express    8328  0.028698
Missing             1577  0.025999

========== card6 ==========
                  count      mean
card6                            
credit           148986  0.066785
Missing            1571  0.024825
debit            439938  0.024263
charge card          15  0.000000
debit or credit      30  0.000000

========== P_emaildomain ==========
                  count      mean
P_emaildomain                    
protonmail.com       76  0.407895
mail.com            559  0.189624
outlook.es          438  0.130137
aim.com             315

In [19]:
data1["card1_freq"] = data1["card1"].map(
    data1["card1"].value_counts()
)

In [20]:
data1["EmailDomainMatch"] = (
    data1["P_emaildomain"] == data1["R_emaildomain"]
).astype(int)

In [21]:
print(
    data1.groupby("EmailDomainMatch")["isFraud"]
    .agg(["count", "mean"])
)

                   count      mean
EmailDomainMatch                  
0                 404644  0.021337
1                 185896  0.064708


In [22]:
data1["P_email_Missing"] = (
    data1["P_emaildomain"] == "Missing"
).astype(int)

data1["R_email_Missing"] = (
    data1["R_emaildomain"] == "Missing"
).astype(int)

In [23]:
data1["CardType"] = (
    data1["card4"].astype(str)
    + "_"
    + data1["card6"].astype(str)
)

In [24]:
print(
    data1.groupby("CardType")["isFraud"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

                               count      mean
CardType                                      
Missing_credit                     3  0.333333
Missing_debit                      9  0.111111
discover_credit                 6304  0.079315
mastercard_credit              50772  0.069152
visa_credit                    83732  0.068122
discover_debit                   347  0.040346
american express_debit           144  0.034722
american express_credit         8175  0.028624
visa_debit                    301023  0.025476
Missing_Missing                 1565  0.024920
mastercard_debit              138415  0.021566
american express_Missing           6  0.000000
american express_charge card       3  0.000000
mastercard_debit or credit        30  0.000000
visa_charge card                  12  0.000000


In [25]:
print("Final shape:", data1.shape)
print("Missing values:", data1.isnull().sum().sum())
print("Duplicate rows:", data1.duplicated().sum())

Final shape: (590540, 119)
Missing values: 0
Duplicate rows: 16


In [26]:
output_path = "../data/processed/feature_engineered.csv"

data1.to_csv(
    output_path,
    index=False
)

print(f"Feature engineered dataset saved to: {output_path}")

Feature engineered dataset saved to: ../data/processed/feature_engineered.csv
